In [17]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, precision_score

# Додати корінь проєкту до шляхів імпорту, щоб бачити модуль src
if ".." not in sys.path:
    sys.path.append("..")

from src.dynamic_pipeline_builder import DynamicPipelineBuilder, validate_config

xlsx_path = "../E Commerce Dataset.xlsx" if os.path.exists("../E Commerce Dataset.xlsx") else "E Commerce Dataset.xlsx"
df = pd.read_excel(xlsx_path, sheet_name="E Comm")
df.columns = df.columns.str.strip()

print(f"Дані успішно завантажено! Розмірність: {df.shape}")
print("Колонки:", df.columns.tolist())

# # 1. Зчитування файлу
# csv_path = "../E_Commerce_Dataset.csv" if os.path.exists("../E_Commerce_Dataset.csv") else "E_Commerce_Dataset.csv"
# df = pd.read_csv(csv_path, sep=None, engine="python")
#
# # Очищення назви колонок від випадкових пробілів
# df.columns = df.columns.str.strip()
#
# print(f"Дані успішно завантажено! Розмірність: {df.shape}")
# print("Фактичні колонки в датасеті:\n", df.columns.tolist())

Дані успішно завантажено! Розмірність: (5630, 20)
Колонки: ['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']


In [18]:
df["y_true"] = df["Churn"].astype(int)

# Евристика: клієнт піде, якщо в нього малий стаж (<= 1) АБО є скарга (Complain == 1)
heuristic_rule = (df["Tenure"] <= 1) | (df["Complain"] == 1)
df["y_heuristic"] = heuristic_rule.astype(int)

acc = accuracy_score(df["y_true"], df["y_heuristic"])
rec = recall_score(df["y_true"], df["y_heuristic"], pos_label=1)
prec = precision_score(df["y_true"], df["y_heuristic"], pos_label=1)

print("-" * 40)
print(f"Accuracy : {acc:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"Precision: {prec:.4f}")
print("-" * 40)

----------------------------------------
Accuracy : 0.6988
Recall   : 0.8481
Precision: 0.3413
----------------------------------------


In [19]:
# Демонстрація перехоплення витоку (CityTier у features)
bad_config = {
    "protected_attributes": ["CityTier", "Gender"],
    "features": {
        "numeric_continuous": ["CityTier", "Tenure"]
    }
}
try:
    validate_config(bad_config)
    print("ПОМИЛКА: Валідатор не зупинив витік!")
except ValueError as e:
    print(f"Успіх: Валідатор перехопив витік захищеної ознаки:\n--> {e}\n")

# Симуляція MCAR пропусків для демонстрації спрацювання різних гілок
df_train = df.copy()
np.random.seed(42)
df_train.loc[np.random.rand(len(df_train)) < 0.20, "CashbackAmount"] = np.nan
df_train.loc[np.random.rand(len(df_train)) < 0.15, "PreferedOrderCat"] = np.nan

# Збирання конвеєра та навчання
config_path = "../config/config_lr.yaml" if os.path.exists("../config/config_lr.yaml") else "config/config_lr.yaml"
builder = DynamicPipelineBuilder(config_path)
pipeline = builder.build()

feature_cols = []
for group in builder.config["features"].values():
    if group: feature_cols.extend(group)

pipeline.fit(df_train[feature_cols])

# Виведення аудиторського сліду
print("--- Аудиторський слід AdaptiveImputer ---")
for name, step, cols in pipeline.transformers_:
    if hasattr(step, "named_steps") and "imputer" in step.named_steps:
        audit = step.named_steps["imputer"].audit_trail_
        for idx, stats in audit.items():
            col_name = cols[idx] if isinstance(idx, int) else idx
            print(f"Колонка: {col_name:<25} | Пропуски: {stats['missing_share']:>6.2%} | Стратегія: {stats['rule']['strategy']}")

Успіх: Валідатор перехопив витік захищеної ознаки:
--> Захищені ознаки потрапили у features: {'CityTier'}

--- Аудиторський слід AdaptiveImputer ---
Колонка: Tenure                    | Пропуски:  4.69% | Стратегія: median
Колонка: WarehouseToHome           | Пропуски:  4.46% | Стратегія: median
Колонка: HourSpendOnApp            | Пропуски:  4.53% | Стратегія: median
Колонка: NumberOfDeviceRegistered  | Пропуски:  0.00% | Стратегія: median
Колонка: NumberOfAddress           | Пропуски:  0.00% | Стратегія: median
Колонка: OrderAmountHikeFromlastYear | Пропуски:  4.71% | Стратегія: median
Колонка: CouponUsed                | Пропуски:  4.55% | Стратегія: median
Колонка: OrderCount                | Пропуски:  4.58% | Стратегія: median
Колонка: DaySinceLastOrder         | Пропуски:  5.45% | Стратегія: median
Колонка: CashbackAmount            | Пропуски: 20.44% | Стратегія: median
Колонка: Complain                  | Пропуски:  0.00% | Стратегія: median
Колонка: PreferredPaymentMode      